In [ ]:
!pip install -q PyPDF2 sentence-transformers chromadb transformers accelerate bitsandbytes gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedder ready ✅")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder ready ✅


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
print("LLM ready ✅")

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LLM ready ✅


In [ ]:
import PyPDF2
import chromadb

chroma_client = chromadb.Client()

def extract_pages(pdf_path):
    reader = PyPDF2.PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append({"page_num": i + 1, "text": text})
    return pages

def chunk_pages(pages, chunk_size=200, overlap=40):
    chunks = []
    for page in pages:
        words = page["text"].split()
        i = 0
        while i < len(words):
            chunk_words = words[i:i+chunk_size]
            chunks.append({"text": " ".join(chunk_words), "page_num": page["page_num"]})
            i += chunk_size - overlap
    return chunks

def build_collection(chunks, collection_name):
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    collection = chroma_client.create_collection(collection_name)
    texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(texts).tolist()
    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=[{"page_num": c["page_num"]} for c in chunks],
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )
    return collection

def retrieve(collection, question, top_k=4):
    q_embedding = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embedding, n_results=top_k)
    return [
        {"text": doc, "page": meta["page_num"]}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]

def ask(collection, question, top_k=4):
    retrieved = retrieve(collection, question, top_k=top_k)
    context = "\n\n".join([f"[Page {r['page']}] {r['text']}" for r in retrieved])

    prompt = f"""<|system|>
You are a helpful assistant. Answer ONLY using the context below.
If the answer is not in the context, say "I couldn't find that in the document."
Always mention which page(s) you used.</s>
<|user|>
Context:
{context}

Question: {question}</s>
<|assistant|>"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.1, do_sample=False)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.split("<|assistant|>")[-1].strip()

print("Pipeline functions ready ✅")

Pipeline functions ready ✅


In [ ]:
# quick sanity check using a manually uploaded PDF
from google.colab import files
uploaded = files.upload()
test_pdf = list(uploaded.keys())[0]

test_pages = extract_pages(test_pdf)
test_chunks = chunk_pages(test_pages)
test_collection = build_collection(test_chunks, "test_collection")

print(ask(test_collection, "What is this document about?"))

Saving Nilgiris_Trip_Plan.pdf to Nilgiris_Trip_Plan.pdf


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


The document appears to be a travel itinerary for a trip to the Nilgiris region in India, specifically for the days of July 6-8, 2019. It includes details about transportation, accommodations, and popular tourist attractions in the area. The document also provides approximate fares and travel times for various modes of transportation, such as TNSTC buses and the Nilgiri Mountain Railway toy train.


In [ ]:
import gradio as gr
import uuid

session_collections = {}

def process_pdf(file, session_id):
    if file is None:
        return "⚠️ Please upload a PDF.", session_id
    if not session_id:
        session_id = str(uuid.uuid4())

    pages = extract_pages(file.name)
    chunks = chunk_pages(pages)
    collection = build_collection(chunks, f"col_{session_id}")
    session_collections[session_id] = collection

    return f"✅ '{file.name}' loaded — {len(pages)} pages, {len(chunks)} chunks. Ask away!", session_id

def chatbot_response(message, history, session_id):
    if not session_id or session_id not in session_collections:
        return "⚠️ Please upload a PDF first (top of the page)."
    return ask(session_collections[session_id], message)

with gr.Blocks(title="📄 PDF Chatbot") as demo:
    gr.Markdown("## 📄 Upload a PDF and chat with it")
    session_id_state = gr.State("")

    pdf_upload = gr.File(label="Upload your PDF", file_types=[".pdf"])
    status = gr.Textbox(label="Status", interactive=False)
    pdf_upload.change(fn=process_pdf, inputs=[pdf_upload, session_id_state], outputs=[status, session_id_state])

    gr.ChatInterface(
        fn=chatbot_response,
        additional_inputs=[session_id_state]
        # examples removed — not needed, and conflicts with additional_inputs
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://82057492b593bae1e9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
